# VisualGuard-LVLM — MME only (download directly in Colab)

**No Google Drive is used.** This notebook downloads the project from GitHub and attempts to download the original MME benchmark files from a public Hugging Face dataset repository directly into the temporary Colab runtime (`/content`). It does **not** rerun POPE.

The notebook validates the downloaded files and converts compatible annotations into the JSONL layout required by this repository. It stops with a diagnostic if the source files do not match the expected schema rather than silently generating misleading results.

> Colab storage is temporary. If the runtime resets, you may need to download the dataset again. Use a GPU runtime. Full MME on a 7B model can take a long time and needs enough GPU memory.

## 1. Check GPU and install download tools

In [ ]:
import os, sys, subprocess, json, time, shutil, zipfile, textwrap
from pathlib import Path

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:2500] or 'No NVIDIA GPU detected yet.')
!pip -q install --upgrade huggingface_hub requests pillow tqdm

Tue Oct 20 07:01:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Download VisualGuard-LVLM directly from GitHub

In [ ]:
REPO = Path('/content/VisualGuard-LVLM')
if not (REPO / 'src' / 'run.py').is_file():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/ria0304/VisualGuard-LVLM.git', str(REPO)],
        check=True
    )
assert (REPO / 'src' / 'run.py').is_file(), f'Project checkout is missing src/run.py: {REPO}'
print('Project ready:', REPO)
print('Commit:', subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip())

Project ready: /content/VisualGuard-LVLM
Commit: 949d9a08499bedbd0920fe3c3306eb9d10ea3d19


## 3. Download MME from the internet into `/content` (no Drive, no manual URL)

The source below is a public Hugging Face mirror that lists the original MME release. Dataset terms still apply. The notebook uses the Hugging Face Hub API to download the dataset files, then searches for the MME release directory.

In [ ]:
from huggingface_hub import snapshot_download

MME_SOURCE_REPO = 'yingss/mixlora-eval-data'
DOWNLOAD_ROOT = Path('/content/datasets/mme_source')
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Downloading MME files from Hugging Face dataset: {MME_SOURCE_REPO}')
source_path = Path(snapshot_download(
    repo_id=MME_SOURCE_REPO,
    repo_type='dataset',
    local_dir=str(DOWNLOAD_ROOT),
    allow_patterns=['mme/**', 'README.md', '.gitattributes'],
    max_workers=8,
))
print('Download location:', source_path)
all_files = [p for p in source_path.rglob('*') if p.is_file()]
print('Downloaded files:', len(all_files))
for p in all_files[:60]:
    print(p.relative_to(source_path), f'({p.stat().st_size:,} bytes)')
if not all_files:
    raise RuntimeError('No MME files were downloaded. Check Colab internet access and the Hugging Face dataset repository.')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2426 files:   0%|          | 0/2426 [00:00<?, ?it/s]

Download location: /content/datasets/mme_source
Downloaded files: 7281
.gitattributes (2,504 bytes)
README.md (4,384 bytes)
mme/MME_Benchmark_release_version/existence.jsonl (11,213 bytes)
mme/MME_Benchmark_release_version/scene.jsonl (84,887 bytes)
mme/MME_Benchmark_release_version/count.jsonl (11,201 bytes)
mme/MME_Benchmark_release_version/OCR.jsonl (7,203 bytes)
mme/MME_Benchmark_release_version/celebrity.jsonl (80,450 bytes)
mme/MME_Benchmark_release_version/artwork.jsonl (80,581 bytes)
mme/MME_Benchmark_release_version/posters.jsonl (59,423 bytes)
mme/MME_Benchmark_release_version/commonsense_reasoning.jsonl (34,225 bytes)
mme/MME_Benchmark_release_version/landmark.jsonl (82,671 bytes)
mme/MME_Benchmark_release_version/position.jsonl (11,747 bytes)
mme/MME_Benchmark_release_version/text_translation.jsonl (10,570 bytes)
mme/MME_Benchmark_release_version/numerical_calculation.jsonl (9,035 bytes)
mme/MME_Benchmark_release_version/code_reasoning.jsonl (8,802 bytes)
mme/MME_Benchmark_

## 4. Normalize MME into the layout VisualGuard expects

In [ ]:
import json, shutil
from pathlib import Path

# VisualGuard expects:
#   <MME_ROOT>/<subtask>/<subtask>.jsonl
#   <MME_ROOT>/<subtask>/images/<image-file>
# Each JSONL row: {"image": <file name inside images/>, "text": <question>, "answer": "Yes"/"No"}
#
# The mirror stores one <task>.jsonl per task directly inside MME_Benchmark_release_version/,
# so the task name is the FILE NAME (ann.stem), not the parent folder name.

# Start clean: earlier attempts wrote every task into one folder called MME_Benchmark_release_version.
MME_ROOT = Path('/content/datasets/MME')
if MME_ROOT.exists():
    shutil.rmtree(MME_ROOT)
MME_ROOT.mkdir(parents=True)

source_root = next(
    p for p in DOWNLOAD_ROOT.rglob('*')
    if p.is_dir() and p.name.lower() == 'mme_benchmark_release_version'
)
print('Source release dir:', source_root)

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
YES, NO = {'yes', 'true', '1'}, {'no', 'false', '0'}

# Index every image in the release by file name.
image_index = {}
for p in source_root.rglob('*'):
    if p.is_file() and p.suffix.lower() in IMG_EXT:
        image_index.setdefault(p.name, []).append(p)
print('Images found in source:', sum(len(v) for v in image_index.values()))

def resolve_image(name, task_dir):
    cands = image_index.get(Path(name).name, [])
    for c in cands:                      # prefer an image inside this task's own folder
        if task_dir in c.parents:
            return c
    return cands[0] if cands else None

summary, missing_total, total_rows = [], 0, 0
for ann in sorted(source_root.glob('*.jsonl')):
    task = ann.stem
    task_src = source_root / task
    out_dir = MME_ROOT / task
    (out_dir / 'images').mkdir(parents=True, exist_ok=True)

    rows, missing = [], []
    for line in ann.read_text(encoding='utf-8', errors='replace').splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        image = obj.get('image', obj.get('image_path', obj.get('image_name', obj.get('image_id'))))
        question = obj.get('text', obj.get('question', obj.get('prompt')))
        answer = obj.get('answer', obj.get('target', obj.get('label')))
        if image is None or question is None or answer is None:
            continue
        a = str(answer).strip().lower()
        if a not in YES | NO:
            continue
        src_img = resolve_image(str(image), task_src)
        if src_img is None:
            missing.append(str(image))
            continue
        link = out_dir / 'images' / src_img.name
        if not link.exists():
            link.symlink_to(src_img)
        rows.append({'image': src_img.name, 'text': str(question),
                     'answer': 'Yes' if a in YES else 'No'})

    (out_dir / f'{task}.jsonl').write_text(
        ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')
    summary.append((task, len(rows), len(missing)))
    total_rows += len(rows)
    missing_total += len(missing)
    if missing:
        print(f'  [{task}] missing images, e.g.:', missing[:3])

print(f'\n{"subtask":28s} rows  missing_images')
for t, n, m in summary:
    print(f'{t:28s} {n:4d}  {m}')
print(f'\nSubtasks: {len(summary)} | Rows: {total_rows} | Missing images: {missing_total}')
print('Normalized MME root:', MME_ROOT)

if len(summary) != 14:
    raise RuntimeError(f'Expected 14 MME subtasks, built {len(summary)}.')
if missing_total:
    raise RuntimeError('Some annotation rows point to images that were not downloaded; see messages above.')


Source release dir: /content/datasets/mme_source/mme/MME_Benchmark_release_version
Images found in source: 1187

subtask                      rows  missing_images
OCR                            40  0
artwork                       400  0
celebrity                     340  0
code_reasoning                 40  0
color                          60  0
commonsense_reasoning         140  0
count                          60  0
existence                      60  0
landmark                      400  0
numerical_calculation          40  0
position                       60  0
posters                       294  0
scene                         400  0
text_translation               40  0

Subtasks: 14 | Rows: 2374 | Missing images: 0
Normalized MME root: /content/datasets/MME


## 4b. Verify images (re-download corrupt ones, drop any that stay unreadable)

In [ ]:
from PIL import Image
from huggingface_hub import hf_hub_download
import json

def find_bad():
    bad = []
    for link in sorted(MME_ROOT.glob('*/images/*')):
        try:
            with Image.open(link) as im:
                im.load()
        except Exception:
            bad.append(link)
    return bad

# 1) Try re-downloading any unreadable image once.
bad = find_bad()
print(f'{len(bad)} unreadable image(s) before repair')
for b in bad:
    try:
        rel = b.resolve().relative_to(DOWNLOAD_ROOT).as_posix()
        hf_hub_download(repo_id=MME_SOURCE_REPO, repo_type='dataset', filename=rel,
                        local_dir=str(DOWNLOAD_ROOT), force_download=True)
    except Exception as e:
        print('  re-download failed for', b.name, ':', e)

# 2) Anything still unreadable is corrupt in the mirror: drop it and its questions.
bad = find_bad()
bad_names = {b.name for b in bad}
dropped = 0
if bad:
    print('Dropping corrupt image(s):', sorted(bad_names))
    for jl in MME_ROOT.glob('*/*.jsonl'):
        rows = [json.loads(l) for l in jl.read_text(encoding='utf-8').splitlines() if l.strip()]
        keep = [r for r in rows if r['image'] not in bad_names]
        if len(keep) != len(rows):
            jl.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in keep), encoding='utf-8')
            dropped += len(rows) - len(keep)
            print(f'  {jl.parent.name}: dropped {len(rows) - len(keep)} rows')
    for b in bad:
        b.unlink()

assert not find_bad(), 'Unreadable images remain.'
total = sum(1 for jl in MME_ROOT.glob('*/*.jsonl') for l in jl.read_text(encoding='utf-8').splitlines() if l.strip())
print(f'All images readable. Questions in use: {total} (excluded from 2374: {dropped})')
if dropped:
    print('NOTE: report that', dropped, 'question(s) were excluded because the mirror image is corrupt.')


1 unreadable image(s) before repair


mme/MME_Benchmark_release_version/landma(…): reconstructing file:   0%|          |  0.00B / 76.9kB            

mme/MME_Benchmark_release_version/landma(…): downloading bytes:           |  0.00B            

Dropping corrupt image(s): ['03d5e3bfc958be38.jpg']
  landmark: dropped 2 rows
All images readable. Questions in use: 2372 (excluded from 2374: 2)
NOTE: report that 2 question(s) were excluded because the mirror image is corrupt.


## 5. Validate dataset and model runtime before running

In [ ]:
%cd /content/VisualGuard-LVLM

import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU: Runtime > Change runtime type > T4 GPU, then rerun from the top.')

# Validate loader discovery and image availability using the project's own loader.
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO) + (os.pathsep + env.get('PYTHONPATH',''))
probe = subprocess.run([sys.executable,'-m','src.run','--help'],cwd=str(REPO),env=env,capture_output=True,text=True,timeout=60)
print('src.run import check:', probe.returncode)
if probe.returncode != 0:
    print((probe.stdout or '')[-1500:]); print((probe.stderr or '')[-2500:])
    raise RuntimeError('src.run import check failed. No model run was started.')

from src.data.mme import discover_subtasks, load_mme
subtasks = discover_subtasks(MME_ROOT)
print('Discovered subtasks:', subtasks)
if len(subtasks) < 14:
    raise RuntimeError(f'Expected the standard 14 MME subtasks; discovered {len(subtasks)}. Do not report this as a full MME benchmark.')
samples = load_mme(MME_ROOT)
print('Validated examples:', len(samples))
print('First sample:', samples[0])
print('Dataset validation passed.')

/content/VisualGuard-LVLM
CUDA available: True
src.run import check: 0
Discovered subtasks: ['existence', 'count', 'position', 'color', 'posters', 'celebrity', 'scene', 'landmark', 'artwork', 'OCR', 'commonsense_reasoning', 'numerical_calculation', 'text_translation', 'code_reasoning']
Validated examples: 2372
First sample: MMESample(question_id='0', subtask='existence', image_path=PosixPath('/content/datasets/MME/existence/images/000000006040.jpg'), question='Is there a train in this image? Please answer yes or no.', answer='Yes', category='perception')
Dataset validation passed.


## 6. Run MME only: baseline and VisualGuard

In [ ]:
# 4-bit quantization (QUANTIZATION = '4bit') requires bitsandbytes; Colab does not preinstall a new enough version.
# src.run runs as a subprocess, so no runtime restart is needed.
!pip -q install -U "bitsandbytes>=0.46.1" accelerate
import importlib.metadata as md
print('bitsandbytes', md.version('bitsandbytes'), '| accelerate', md.version('accelerate'))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 37.1 MB/s eta 0:00:00
bitsandbytes 0.50.2 | accelerate 1.15.0


In [9]:
# Match the model settings used in your POPE experiment.
MODEL = 'llava-hf/llava-1.5-7b-hf'
DEVICE = 'cuda'
DTYPE = 'float16'
QUANTIZATION = '4bit'
SEED = 42

# None = full dataset. Set an integer only for a pilot, and do not report it as a full result.
MAX_SAMPLES = None

RESULTS = Path('/content/VisualGuard-LVLM/results/mme_only')
LOGS = RESULTS / 'logs'
RESULTS.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)

common = [
    '--benchmark','mme', '--model',MODEL, '--device',DEVICE,
    '--dtype',DTYPE, '--quantization',QUANTIZATION,
    '--data-root',str(MME_ROOT), '--output-dir',str(RESULTS),
    '--seed',str(SEED), '--log-level','INFO'
]
if MAX_SAMPLES is not None:
    common += ['--max-samples',str(MAX_SAMPLES)]

def run_mme(method, config):
    name=f'mme_{method}_s{SEED}'
    result_path=RESULTS/f'{name}.json'
    log_path=LOGS/f'{name}.log'
    if result_path.exists() and result_path.stat().st_size > 0:
        print(f'[SKIP] Existing result found: {result_path}')
        return
    cmd=[sys.executable,'-m','src.run',*common,'--method',method,'--config',config,'--run-name',name]
    print('\nRUN:', ' '.join(cmd))
    start=time.time()
    with log_path.open('w',encoding='utf-8') as log:
        proc=subprocess.Popen(cmd,cwd=str(REPO),env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end='')
            log.write(line); log.flush()
        rc=proc.wait()
    print(f'Finished {name}: exit={rc}, elapsed={(time.time()-start)/60:.1f} min')
    if rc != 0:
        raise RuntimeError(f'{name} failed; inspect {log_path}. Fix the error before continuing.')
    if not result_path.exists():
        print('WARNING: process succeeded but expected result file was not found:', result_path)

run_mme('baseline','baseline')
run_mme('visualguard','visualguard')
print('MME run cell complete. Results:', RESULTS)


RUN: /usr/bin/python3 -m src.run --benchmark mme --model llava-hf/llava-1.5-7b-hf --device cuda --dtype float16 --quantization 4bit --data-root /content/datasets/MME --output-dir /content/VisualGuard-LVLM/results/mme_only --seed 42 --log-level INFO --method baseline --config baseline --run-name mme_baseline_s42
2026-10-20 07:05:46,145 INFO    visualguard: Method: baseline
2026-10-20 07:05:46,145 INFO    visualguard: LVLM: llava-hf/llava-1.5-7b-hf
2026-10-20 07:05:46,145 INFO    visualguard: Evidence weights: alpha=0.0 beta=0.0 gamma=0.0 lambda=0.0 threshold=1.0
2026-10-20 07:05:46,145 INFO    visualguard: MME subtasks discovered: ['existence', 'count', 'position', 'color', 'posters', 'celebrity', 'scene', 'landmark', 'artwork', 'OCR', 'commonsense_reasoning', 'numerical_calculation', 'text_translation', 'code_reasoning']
2026-10-20 07:05:54,973 INFO    numexpr.utils: NumExpr defaulting to 2 threads.
2026-10-20 07:05:55,439 INFO    src.model.llava_backend: Loading LVLM llava-hf/llava-1

## 7. Inspect results and logs

In [10]:
result_files=sorted(RESULTS.glob('*.json'))
print('Result files:')
for p in result_files:
    print('\n---',p.name, f'({p.stat().st_size:,} bytes) ---')
    try:
        data=json.loads(p.read_text(encoding='utf-8'))
        print(json.dumps(data,indent=2)[:7000])
    except Exception as e:
        print('Could not parse:',e)
print('\nLogs:')
for p in sorted(LOGS.glob('*.log')):
    print(p, f'({p.stat().st_size:,} bytes)')
print('All files are in temporary Colab storage under /content; download results before disconnecting.')

Result files:

--- mme_baseline_s42.json (5,900 bytes) ---
{
  "efficiency": {
    "peak_gpu_memory_mb": 4414.91015625,
    "total_runtime_s": 2203.4649207390003
  },
  "mme": {
    "benchmark": "mme",
    "category_totals": {
      "cognition_score": 310.3571428571429,
      "cognition_subtasks_measured": 4,
      "perception_score": 1466.021937920897,
      "perception_subtasks_measured": 10,
      "total_score": 1776.37908077804
    },
    "latency_s": 2011.384462455,
    "method": "baseline",
    "metrics": {
      "OCR/accuracy": 82.5,
      "OCR/accuracy_plus": 65.0,
      "OCR/n_images": 20,
      "OCR/n_questions": 40,
      "OCR/score": 147.5,
      "artwork/accuracy": 70.0,
      "artwork/accuracy_plus": 45.0,
      "artwork/n_images": 200,
      "artwork/n_questions": 400,
      "artwork/score": 114.99999999999999,
      "celebrity/accuracy": 78.82352941176471,
      "celebrity/accuracy_plus": 61.1764705882353,
      "celebrity/n_images": 170,
      "celebrity/n_questions": 

In [ ]:
# CELL 1 — MME ablations + sensitivity analysis
# Uses the same model, MME dataset, seed, device, and quantization as your existing run.
# Efficiency (runtime and peak GPU memory) is recorded automatically by src.run.

import re
import subprocess
import time
from pathlib import Path

EXPERIMENTS = [
    # Component ablations
    ("ablation_attention", "attention", "attention", []),
    ("ablation_semantic", "semantic", "semantic", []),
    ("ablation_attention_semantic", "unidirectional", "unidirectional", []),

    # Sensitivity: change one parameter at a time from the default configuration
    ("sensitivity_lambda_025", "visualguard", "visualguard", ["--lambda", "0.25"]),
    ("sensitivity_lambda_075", "visualguard", "visualguard", ["--lambda", "0.75"]),
    ("sensitivity_threshold_025", "visualguard", "visualguard", ["--threshold", "0.25"]),
    ("sensitivity_threshold_045", "visualguard", "visualguard", ["--threshold", "0.45"]),
]

# Preserve your current settings, but give each run its own output directory
# so prediction files from different experiments cannot overwrite each other.
base_args = common.copy()
for flag in ["--output-dir"]:
    if flag in base_args:
        i = base_args.index(flag)
        del base_args[i:i+2]

for run_name, method, config, overrides in EXPERIMENTS:
    run_dir = RESULTS / run_name
    result_file = run_dir / f"{run_name}.json"
    log_file = LOGS / f"{run_name}.log"
    run_dir.mkdir(parents=True, exist_ok=True)

    if result_file.exists() and result_file.stat().st_size > 0:
        print(f"[SKIP] Existing result: {result_file}")
        continue

    cmd = [
        sys.executable, "-m", "src.run",
        *base_args,
        "--output-dir", str(run_dir),
        "--method", method,
        "--config", config,
        "--run-name", run_name,
        *overrides,
    ]

    print(f"\n{'='*70}\nRUNNING: {run_name}\n{' '.join(cmd)}\n{'='*70}")
    started = time.time()

    with log_file.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(
            cmd, cwd=str(REPO), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        code = proc.wait()

    print(f"Finished {run_name} | exit={code} | elapsed={(time.time()-started)/60:.1f} min")
    if code != 0:
        raise RuntimeError(f"{run_name} failed. Inspect log: {log_file}")

print("\nExperiment sweep finished. Results are under:", RESULTS)


RUNNING: ablation_attention
/usr/bin/python3 -m src.run --benchmark mme --model llava-hf/llava-1.5-7b-hf --device cuda --dtype float16 --quantization 4bit --data-root /content/datasets/MME --seed 42 --log-level INFO --output-dir /content/VisualGuard-LVLM/results/mme_only/ablation_attention --method attention --config attention --run-name ablation_attention
2026-10-20 08:58:42,964 INFO    visualguard: Method: attention
2026-10-20 08:58:42,964 INFO    visualguard: LVLM: llava-hf/llava-1.5-7b-hf
2026-10-20 08:58:42,964 INFO    visualguard: Evidence weights: alpha=1.0 beta=0.0 gamma=0.0 lambda=0.5 threshold=0.35
2026-10-20 08:58:42,965 INFO    visualguard: MME subtasks discovered: ['existence', 'count', 'position', 'color', 'posters', 'celebrity', 'scene', 'landmark', 'artwork', 'OCR', 'commonsense_reasoning', 'numerical_calculation', 'text_translation', 'code_reasoning']
2026-10-20 08:58:48,095 INFO    numexpr.utils: NumExpr defaulting to 2 threads.
2026-10-20 08:58:48,369 INFO    src.mo

In [7]:

from pathlib import Path

for p in [
    Path("/content/VisualGuard-LVLM"),
    Path("/content/datasets/MME"),
    Path("/content/drive"),
]:
    print(f"{p}: {'EXISTS' if p.exists() else 'MISSING'}")

print("\nTop-level /content contents:")
for p in Path("/content").iterdir():
    print(p)


/content/VisualGuard-LVLM: MISSING
/content/datasets/MME: MISSING
/content/drive: MISSING

Top-level /content contents:
/content/.config
/content/VisualGuard_LVLM_MME_completed.zip
/content/sample_data


In [5]:
# CELL 2 — Collect experiment results and produce paper-ready summaries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

all_json = [
    p for p in RESULTS.rglob("*.json")
    if p.is_file()
]

rows = []
for path in sorted(all_json):
    try:
        d = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue

    run_name = d.get("provenance", {}).get("run_name", path.stem)
    efficiency = d.get("efficiency", {})
    row = {
        "run": run_name,
        "benchmark": "MME" if "mme" in d else "POPE" if "pope" in d else "other",
        "runtime_min": round(efficiency.get("total_runtime_s", 0) / 60, 2),
        "peak_GPU_memory_MB": efficiency.get("peak_gpu_memory_mb"),
        "MME_total_score": d.get("mme", {}).get("category_totals", {}).get("total_score"),
        "MME_perception": d.get("mme", {}).get("category_totals", {}).get("perception_score"),
        "MME_cognition": d.get("mme", {}).get("category_totals", {}).get("cognition_score"),
    }

    # POPE may contain one or more settings (random/popular/adversarial).
    if "pope" in d:
        for setting, block in d["pope"].items():
            m = block.get("metrics", {})
            rows.append({
                **row,
                "run": f"{run_name} / {setting}",
                "POPE_accuracy": m.get("accuracy"),
                "POPE_F1": m.get("f1"),
                "POPE_hallucination_rate": m.get("hallucination_rate"),
                "samples": m.get("n", block.get("n_samples")),
            })
    elif "mme" in d:
        rows.append({**row, "samples": d["mme"].get("n_samples")})

summary = pd.DataFrame(rows)

if summary.empty:
    print("No experiment JSON files found in:", RESULTS)
else:
    print("Experiment summary")
    display(summary)

    summary_path = RESULTS / "paper_results_summary.csv"
    summary.to_csv(summary_path, index=False)
    print("Saved table:", summary_path)

    # MME official total score comparison
    mme = summary[
        (summary["benchmark"] == "MME") &
        summary["MME_total_score"].notna()
    ].drop_duplicates(subset=["run"])

    if not mme.empty:
        plt.figure(figsize=(12, 5))
        plt.bar(mme["run"], mme["MME_total_score"])
        plt.ylabel("MME total score")
        plt.title("VisualGuard-LVLM: MME experiment comparison")
        plt.xticks(rotation=40, ha="right")
        plt.tight_layout()
        plt.show()

    # POPE metrics comparison
    pope = summary[summary["benchmark"] == "POPE"].copy()
    if not pope.empty and "POPE_F1" in pope.columns:
        pope = pope[pope["POPE_F1"].notna()]
        if not pope.empty:
            plt.figure(figsize=(12, 5))
            plt.bar(pope["run"], pope["POPE_F1"])
            plt.ylabel("F1 score")
            plt.title("POPE F1 comparison")
            plt.xticks(rotation=40, ha="right")
            plt.tight_layout()
            plt.show()

# Paired bootstrap confidence interval for baseline vs VisualGuard MME
# Uses per-question correctness on matched question IDs.
def load_mme_predictions(filename):
    candidates = list(RESULTS.rglob(filename))
    if not candidates:
        return None

    # Prefer the direct results folder if it contains the prediction file.
    direct = RESULTS / filename
    path = direct if direct.exists() else candidates[0]

    records = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                records[str(r["question_id"])] = int(bool(r["correct"]))
    return records

baseline = load_mme

No experiment JSON files found in: /content/VisualGuard-LVLM/results/mme_only


NameError: name 'load_mme' is not defined